In [ ]:
# -*- coding: utf-8 -*-
"""
Created on 2026-06-05
Revised on 2026-06-05

@author:       Oscar Trevizo
@institution:  Harvard Extension School — Graduate Data Science Program (2023)
@context:      Independent project — applying course concepts to real-world data
@environment:  Python 3.14.3 | myenv | MacBook Air M5

Hello World of Agentic AI — Stock Tracking Agent
=================================================

Description:
    A minimal but complete Agentic AI implementation using the Anthropic Claude
    API with Tool Use. The agent answers natural-language questions about stock
    prices by calling Python tools backed by yfinance.

    This is the 'Hello World' of Agentic AI:
      - Define tools (functions Claude can call)
      - Run the agent loop (Claude reasons → calls tools → reasons → responds)
      - Ask questions in plain English, get real market data back

Key Concepts:
    Tool Use     — Claude decides when and which tools to call
    Agent Loop   — a cycle: send message → get tool call → execute → feed result back
    Grounding    — Claude's answers are grounded in live data, not training knowledge

Revision History:
    2026-06-05  Original development
                - get_stock_price and get_portfolio_summary tools
                - Single-question agent and multi-turn chat session
"""

## Cell [2] — Prerequisites

**Install libraries** (one-time, in myenv):
```bash
pip install anthropic yfinance python-dotenv
```

**Set your API key** — create a `.env` file in this folder (it is git-ignored):
```
# /Users/otrevizo/GitHub/Python/agentic_ai/.env
ANTHROPIC_API_KEY=sk-ant-...
```

Get your key at: https://console.anthropic.com

In [ ]:
# Cell [2] — Imports
import os
import json
from datetime import datetime
from pathlib import Path

import yfinance as yf
import anthropic
from dotenv import load_dotenv

# Load API key from .env file in this folder (falls back to shell env var)
load_dotenv(Path(__file__).parent / '.env' if '__file__' in dir() else '.env')

api_key = os.environ.get('ANTHROPIC_API_KEY')
if not api_key:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY not set.\n"
        "Option 1: create agentic_ai/.env with ANTHROPIC_API_KEY=sk-ant-...\n"
        "Option 2: export ANTHROPIC_API_KEY='sk-ant-...' in your terminal, then restart kernel."
    )

client = anthropic.Anthropic(api_key=api_key)
print(f"Anthropic client ready. SDK version: {anthropic.__version__}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Cell [3] — Tool Definitions (Python side)

These are the actual Python functions that fetch data.
Claude does not run these — the **agent loop** (Cell 5) runs them
when Claude asks for them.

In [ ]:
# Cell [3] — Python tool implementations

def get_stock_price(ticker: str) -> dict:
    """
    Fetch current price and key stats for a single stock ticker.

    Parameters
    ----------
    ticker : str
        Stock symbol, e.g. 'AAPL', 'NVDA', 'MSFT'

    Returns
    -------
    dict
        price, change, change_pct, 52w_high, 52w_low, volume, as_of
        On error: {ticker, error}
    """
    try:
        ticker = ticker.upper().strip()
        stock = yf.Ticker(ticker)
        info = stock.fast_info

        current_price = info.last_price
        prev_close    = info.previous_close
        change        = current_price - prev_close
        change_pct    = (change / prev_close) * 100

        return {
            "ticker":     ticker,
            "price":      round(current_price, 2),
            "change":     round(change, 2),
            "change_pct": round(change_pct, 2),
            "52w_high":   round(info.fifty_two_week_high, 2),
            "52w_low":    round(info.fifty_two_week_low, 2),
            "volume":     int(info.three_month_average_volume or 0),
            "as_of":      datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
    except Exception as e:
        return {"ticker": ticker, "error": str(e)}


def get_portfolio_summary(tickers: list) -> list:
    """
    Fetch price data for a list of stock tickers.

    Parameters
    ----------
    tickers : list of str
        E.g. ['AAPL', 'NVDA', 'MSFT']

    Returns
    -------
    list of dict
        One result dict per ticker (same schema as get_stock_price).
    """
    return [get_stock_price(t) for t in tickers]


print("Tools defined: get_stock_price, get_portfolio_summary")

# Sanity check — hits yfinance directly, no Claude involved yet
test = get_stock_price('AAPL')
print(f"Sanity check — AAPL: ${test.get('price')} ({test.get('change_pct')}%)")

## Cell [4] — Tool Schema (Claude's side)

JSON descriptions sent to Claude so it knows the tools exist
and understands their parameters. Claude reads these to decide
when and how to call each tool.

In [ ]:
# Cell [4] — Tool schemas (sent to Claude with every API call)

TOOLS = [
    {
        "name": "get_stock_price",
        "description": (
            "Get the current stock price and key statistics for a single ticker symbol. "
            "Returns price, daily change, change percent, 52-week high/low, and volume. "
            "Use this when the user asks about a specific stock."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol, e.g. 'AAPL', 'NVDA', 'MSFT', 'TSLA'"
                }
            },
            "required": ["ticker"]
        }
    },
    {
        "name": "get_portfolio_summary",
        "description": (
            "Get current prices for a list of stock tickers at once. "
            "Use this when the user asks about multiple stocks or their whole watchlist."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "tickers": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of ticker symbols, e.g. ['AAPL', 'NVDA', 'MSFT']"
                }
            },
            "required": ["tickers"]
        }
    }
]

print(f"{len(TOOLS)} tools registered with Claude.")

## Cell [5] — The Agent Loop

The heart of Agentic AI. The loop:

1. Send user message + tool schemas to Claude
2. Claude responds — with a final answer **or** a `tool_use` request
3. If `tool_use`: execute the tool in Python, feed the result back to Claude
4. Repeat until Claude gives a final text response

```
User → Claude → [tool_use?] → Python → Claude → Final answer
                    ↑________________________________|
                            (loop until done)
```

In [ ]:
# Cell [5] — Agent loop

# Dispatch table: maps tool name → Python function
TOOL_FUNCTIONS = {
    "get_stock_price":      get_stock_price,
    "get_portfolio_summary": get_portfolio_summary,
}

SYSTEM_PROMPT = """
You are a concise, data-driven stock tracking assistant.
When given stock data, present it clearly: ticker, price, daily change ($ and %),
and a one-sentence interpretation. Mention if a stock is near its 52-week high
or low when relevant. Be brief — the user is analytically sophisticated.
"""


def run_agent(user_question: str, verbose: bool = True) -> str:
    """
    Run the Claude stock agent on a single question.

    The agent loop:
      1. Send question to Claude with tool schemas
      2. If Claude calls a tool: execute it, return result, loop
      3. When Claude returns text: done

    Parameters
    ----------
    user_question : str
        Natural language question about stocks.
    verbose : bool
        Print tool calls as they happen (good for learning the pattern).

    Returns
    -------
    str
        Claude's final response.
    """
    messages = [{"role": "user", "content": user_question}]
    max_iterations = 10  # safety guard against infinite loops

    for iteration in range(max_iterations):

        # --- Step 1: Call Claude ---
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",  # fast + cost-effective for tool use
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages
        )

        # --- Step 2a: Final answer ---
        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, 'text'):
                    return block.text

        # --- Step 2b: Tool call ---
        elif response.stop_reason == "tool_use":
            # Append Claude's response (including tool_use blocks) to history
            messages.append({"role": "assistant", "content": response.content})

            # Execute each tool Claude requested
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  [Tool call #{iteration+1}] {block.name}({block.input})")

                    tool_fn = TOOL_FUNCTIONS.get(block.name)
                    result  = tool_fn(**block.input) if tool_fn else {"error": f"Unknown tool: {block.name}"}

                    tool_results.append({
                        "type":        "tool_result",
                        "tool_use_id": block.id,
                        "content":     json.dumps(result)
                    })

            # Feed results back to Claude
            messages.append({"role": "user", "content": tool_results})

        else:
            return f"Unexpected stop_reason: {response.stop_reason}"

    return "Max iterations reached without a final answer."


print("Agent loop ready — run_agent() defined.")

## Cell [6] — Run the Agent

Ask anything in plain English. Claude will decide which tool(s) to call.

In [ ]:
# Cell [6] — Single stock question

question = "How is Apple doing today?"

print(f"Q: {question}")
print("-" * 50)
answer = run_agent(question)
print(f"\nAgent: {answer}")

In [ ]:
# Cell [7] — Portfolio watchlist
# Edit MY_WATCHLIST with your actual tickers

MY_WATCHLIST = ['AAPL', 'NVDA', 'MSFT', 'GOOGL', 'META']

question = f"Give me a summary of my watchlist: {', '.join(MY_WATCHLIST)}"

print(f"Q: {question}")
print("-" * 50)
answer = run_agent(question)
print(f"\nAgent: {answer}")

In [ ]:
# Cell [8] — Multi-turn chat session
# The agent remembers context across turns within the session.
# Try: "How is NVDA?" then "Compare it to AMD" (no need to repeat the ticker)

def chat_session():
    """
    Interactive multi-turn chat with the stock agent.
    Maintains full conversation history across turns.
    Type 'quit' to exit.
    """
    messages = []
    print("Stock Agent — multi-turn chat. Type 'quit' to exit.")
    print("=" * 50)

    while True:
        user_input = input("\nYou: ").strip()
        if user_input.lower() in ('quit', 'exit', 'q'):
            print("Session ended.")
            break
        if not user_input:
            continue

        messages.append({"role": "user", "content": user_input})

        for _ in range(10):
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=1024,
                system=SYSTEM_PROMPT,
                tools=TOOLS,
                messages=messages
            )

            if response.stop_reason == "end_turn":
                for block in response.content:
                    if hasattr(block, 'text'):
                        print(f"\nAgent: {block.text}")
                        messages.append({"role": "assistant", "content": block.text})
                break

            elif response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        print(f"  [fetching {block.name}({block.input})...]")
                        fn     = TOOL_FUNCTIONS.get(block.name)
                        result = fn(**block.input) if fn else {"error": "unknown tool"}
                        tool_results.append({
                            "type":        "tool_result",
                            "tool_use_id": block.id,
                            "content":     json.dumps(result)
                        })
                messages.append({"role": "user", "content": tool_results})


# Uncomment to run interactive chat:
# chat_session()

## Next Steps

Now that the 'Hello World' agent works, natural extensions:

1. **Currency tool** — `get_fx_rate('USD', 'EUR')` via yfinance FX tickers (`EURUSD=X`, `USDMXN=X`)
2. **News tool** — fetch recent headlines per ticker (newsapi or yfinance `.news`)
3. **Persist watchlist** — store tickers + conversation history in JSON or SQLite
4. **Morning briefing** — schedule a daily run that logs or emails a summary
5. **Multi-agent** — one agent fetches data, a second writes the narrative analysis

---
*Built with Anthropic Claude API + yfinance*  
*Oscar Trevizo — GitHub: otrevizo | Harvard DS Program (2023)*